In [1]:
import numpy as np
import rasterio as rio
from rasterio.plot import show
import pandas as pd
from pathlib import Path

In [2]:
# Create markers if necessary, otherwise mute. 

satfolder = "/home/jovyan/work/AVOCA/GEM/Development/Data/BatchExport_LST/YYYY/GEMLST_MODIS_yyyymmdd.tif"

# ERA5 Data: 
imfolder = "/home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL1000mCal"
imfiles = sorted(Path(imfolder).glob("*.tif"))

# markers (adjust if you need case-insensitive match or different substrings)
start_marker = "t2m_rc_2016_d001.tif" # Missing: t2m_rc_2007_d357.tif
end_marker = "t2m_rc_2020_d058.tif" # 27.02.2020 = start orbital drift terra

# find first index containing the start marker and last index containing the end marker
start_idx = next((i for i, p in enumerate(imfiles) if start_marker in p.name), None)
end_idx = next((i for i, p in enumerate(imfiles) if end_marker in p.name), None)

if start_idx is None:
    raise FileNotFoundError(f"No file containing '{start_marker}' found under {imfolder}")
if end_idx is None:
    raise FileNotFoundError(f"No file containing '{end_marker}' found under {imfolder}")
if end_idx < start_idx:
    raise ValueError(f"End file '{end_marker}' appears before start file '{start_marker}' in sorted file order")

# slice inclusive range
imfiles = imfiles[start_idx:end_idx + 1]

In [ ]:
for imfile in imfiles:

    # Find and match dates
    yearstring = imfile.stem.split("_")[2]
    year = pd.to_datetime(yearstring, format="%Y")
    doy = imfile.stem.split("_")[3][1:]
    date = year + pd.to_timedelta(int(doy) - 1, unit="D")
    date = date.strftime("%Y-%m-%d")
    datestring = date.replace("-", "")

    imagepath = satfolder.replace('YYYY', yearstring).replace('yyyymmdd', datestring)
    modis = rio.open(imagepath)
    era5 = rio.open(imfile)

    qa = modis.read(2)

    gapfilled = np.where(qa == 0, era5.read(1), modis.read(1))
        
    # Save gapfilled as new GeoTIFF including the original qa band (qa)
    output_path = f"/home/jovyan/work/AVOCA/GEM/Production/Gapfilled/GEMLST_{datestring}.tiff"
    with rio.open(
        output_path,
        "w",
        driver="GTiff",
        height=gapfilled.shape[0],
        width=gapfilled.shape[1],
        count=2,
        dtype=np.float32,
        crs=modis.crs,
        transform=modis.transform,
    ) as dst:
        dst.write(gapfilled, 1)
        dst.write(qa, 2)

    print(f'GEMLST_{datestring} done')

print(f'years {start_marker.split('_',3)[2]} - {end_marker.split('_',3)[2]} done')


GEMLST_20160101 done
GEMLST_20160102 done
GEMLST_20160103 done
GEMLST_20160104 done
GEMLST_20160105 done
GEMLST_20160106 done
